## 14 Takeout Demand & Food Coverage

**笔记本**：`14_takeout_demand_coverage.ipynb`

**输入**：
- `../08 POI Demand/data_out/sz_demand_grid.gpkg`（H3 res8 格网 + POI 计数）
- `../08 POI Demand/data_out/sz_poi.gpkg`（478K POI 点位，含 food 类型）
- `../09 Population/data_out/sz_population_grid.gpkg`（人口 + 住宅 + 小区）
- `../12 RL-Dispatch/data_raw/RL-Dispatch_SZ_order.csv`（150 万真实配送订单 → 空间聚合）
- `../12 RL-Dispatch/data_out/dispatch_demand_compare.csv`（ground truth 对照）

**做了什么 / 算了什么**：
1. **真实订单空间聚合**：12 RL-Dispatch 的 receiver_lat/lon → H3 空间连接 → `real_order_count` / `real_order_density` 每格真实订单量。
2. **`takeout_demand_index`**：融合 real_order_count (0.50) + pop_count (0.30) + residential_count (0.20)。真实订单为主，人口 + 居住建筑为补。food_count 剥离到 supply 层避免自相关。
3. **`food_access_1km / 2km / 3km`**：对每个 H3 格网质心，用 cKDTree 计算 1/2/3 km 半径内的餐饮 POI 数量 → 外卖覆盖率/可达性。
4. **Ground Truth 验证**：将指标与 12 的 dispatch_orders 做相关分析。

**时间维度**：前端使用 13 Meituan `meituan_hourly_demand.json` 的经典午/晚双峰时序。

**写出文件**：
- `data_out/sz_takeout_demand_grid.gpkg`

In [ ]:
from pathlib import Path
import geopandas as gpd
import pandas as pd
import numpy as np
from scipy.spatial import cKDTree
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

OUT = Path("data_out")
OUT.mkdir(exist_ok=True)

DEMAND_GRID = Path("../08 POI Demand/data_out/sz_demand_grid.gpkg")
POI_POINTS  = Path("../08 POI Demand/data_out/sz_poi.gpkg")
POP_GRID    = Path("../09 Population/data_out/sz_population_grid.gpkg")
RL_CSV      = Path("../12 RL-Dispatch/data_raw/RL-Dispatch_SZ_order.csv")
DISPATCH    = Path("../12 RL-Dispatch/data_out/dispatch_demand_compare.csv")

PROJ_METRIC = "EPSG:32650"
CHUNK_ROWS  = 200_000

for label, p in [("Demand grid", DEMAND_GRID), ("POI points", POI_POINTS),
                 ("Pop grid", POP_GRID), ("RL CSV", RL_CSV), ("Dispatch", DISPATCH)]:
    print(f"{label}: {'OK' if p.exists() else 'MISSING'}  {p}")

In [ ]:
# ============================================================
#  1. Load grids & POI points
# ============================================================

demand = gpd.read_file(DEMAND_GRID).to_crs(4326)
pop = gpd.read_file(POP_GRID).to_crs(4326)
poi_all = gpd.read_file(POI_POINTS).to_crs(4326)

print(f"Demand grid: {len(demand):,} cells")
print(f"Population grid: {len(pop):,} cells")
print(f"POI total: {len(poi_all):,} points")

food_poi = poi_all[poi_all["poi_type"] == "food"].copy()
print(f"Food POI: {len(food_poi):,} points")

grid = demand[["h3_id", "food_count", "poi_total", "demand_pressure", "geometry"]].copy()

pop_cols = ["h3_id", "pop_count", "residential_count", "xiaoqu_count",
            "intensity_index", "pop_density"]
pop_cols = [c for c in pop_cols if c in pop.columns]
grid = grid.merge(pop[pop_cols], on="h3_id", how="left")

for c in ["pop_count", "residential_count", "xiaoqu_count"]:
    if c not in grid.columns:
        grid[c] = 0
    grid[c] = grid[c].fillna(0)

print(f"\nMerged grid: {len(grid):,} cells")

In [ ]:
# ============================================================
#  2. Real Order Spatial Aggregation (12 RL-Dispatch)
#     receiver_lat/lon → H3 grid → real_order_count
# ============================================================

print("Reading 12 RL-Dispatch orders in chunks ...")
order_frames = []
total_raw = 0

for chunk in pd.read_csv(RL_CSV, chunksize=CHUNK_ROWS):
    total_raw += len(chunk)
    for c in ["receiver_latitude", "receiver_longitude", "order_cnt"]:
        chunk[c] = pd.to_numeric(chunk[c], errors="coerce")
    chunk = chunk.dropna(subset=["receiver_latitude", "receiver_longitude"])
    chunk = chunk[
        (chunk["receiver_latitude"].between(22.35, 22.95)) &
        (chunk["receiver_longitude"].between(113.65, 114.75))
    ]
    chunk["order_cnt"] = chunk["order_cnt"].fillna(1).clip(lower=1)
    order_frames.append(chunk[["receiver_latitude", "receiver_longitude", "order_cnt", "send_hour"]])
    print(f"  read {total_raw:>10,} rows, kept {sum(len(f) for f in order_frames):,}", end="\r")

orders = pd.concat(order_frames, ignore_index=True)
print(f"\nTotal raw: {total_raw:,}, filtered: {len(orders):,}, orders: {orders['order_cnt'].sum():,.0f}")

# Spatial join to H3 grid
order_pts = gpd.GeoDataFrame(
    orders,
    geometry=gpd.points_from_xy(orders["receiver_longitude"], orders["receiver_latitude"]),
    crs=4326,
)
joined = gpd.sjoin(order_pts, grid[["h3_id", "geometry"]], how="inner", predicate="within")
print(f"Spatially matched: {len(joined):,}")

order_agg = joined.groupby("h3_id").agg(
    real_order_count=("order_cnt", "sum"),
).reset_index()

# 24h hourly profile from 12 RL
hourly_rl = joined.groupby("send_hour")["order_cnt"].sum().reindex(range(24), fill_value=0)
print(f"\nGrids with real orders: {len(order_agg):,}")
print(f"Total real orders aggregated: {order_agg['real_order_count'].sum():,.0f}")

grid = grid.merge(order_agg, on="h3_id", how="left")
grid["real_order_count"] = grid["real_order_count"].fillna(0)
grid["real_order_density"] = grid["real_order_count"] / grid.geometry.to_crs(PROJ_METRIC).area * 1e6

print(f"\nreal_order_count: max={grid['real_order_count'].max():,.0f}, "
      f"mean={grid['real_order_count'].mean():,.1f}, "
      f"non-zero={( grid['real_order_count'] > 0).sum():,}/{len(grid):,}")

In [ ]:
# ============================================================
#  3. Takeout Demand Index (融合真实订单 + 人口 + 设施)
#     real_order(0.35) + pop(0.25) + xiaoqu(0.15) + food(0.15) + res(0.10)
# ============================================================

def safe_normalize(s):
    mn, mx = s.min(), s.max()
    if mx == mn:
        return pd.Series(0.0, index=s.index)
    return (s - mn) / (mx - mn)

grid["order_norm"] = safe_normalize(grid["real_order_count"])
grid["pop_norm"] = safe_normalize(grid["pop_count"])
grid["xiaoqu_norm"] = safe_normalize(grid["xiaoqu_count"])
grid["food_norm"] = safe_normalize(grid["food_count"])
grid["res_norm"] = safe_normalize(grid["residential_count"])

W_ORDER, W_POP, W_XIAOQU, W_FOOD, W_RES = 0.35, 0.25, 0.15, 0.15, 0.10

grid["takeout_demand_index"] = (
    W_ORDER  * grid["order_norm"]
  + W_POP    * grid["pop_norm"]
  + W_XIAOQU * grid["xiaoqu_norm"]
  + W_FOOD   * grid["food_norm"]
  + W_RES    * grid["res_norm"]
)

grid["takeout_demand_norm"] = safe_normalize(grid["takeout_demand_index"])

print("=== Takeout Demand Index (with real orders) ===")
print(f"Weights: order={W_ORDER}, pop={W_POP}, xiaoqu={W_XIAOQU}, food={W_FOOD}, res={W_RES}")
print(f"Range: {grid['takeout_demand_index'].min():.4f} ~ {grid['takeout_demand_index'].max():.4f}")
print(f"Mean:  {grid['takeout_demand_index'].mean():.4f}")
print(f"Non-zero: {(grid['takeout_demand_index'] > 0).sum():,} / {len(grid):,}")
print(f"\nTop 10:")
print(grid.nlargest(10, "takeout_demand_index")[
    ["h3_id", "real_order_count", "pop_count", "xiaoqu_count", "food_count",
     "residential_count", "takeout_demand_index"]
].to_string(index=False))

In [ ]:
# ============================================================
#  4. Food Accessibility Buffer (cKDTree)
#     每个格网质心 1/2/3 km 内能吃到多少家餐厅
# ============================================================

food_m = food_poi.to_crs(PROJ_METRIC)
food_coords = np.column_stack([food_m.geometry.x, food_m.geometry.y])
food_tree = cKDTree(food_coords)
print(f"Food cKDTree built: {len(food_coords):,} points")

grid_m = grid.to_crs(PROJ_METRIC)
centroids = np.column_stack([
    grid_m.geometry.centroid.x,
    grid_m.geometry.centroid.y
])

RADII = [1000, 2000, 3000]
for radius in RADII:
    col = f"food_access_{radius // 1000}km"
    hits = food_tree.query_ball_point(centroids, r=radius)
    grid[col] = [len(h) for h in hits]
    nz = (grid[col] > 0).sum()
    print(f"  {col}: max={grid[col].max():,}, mean={grid[col].mean():.1f}, non-zero={nz:,}/{len(grid):,}")

grid["food_access_norm"] = safe_normalize(grid["food_access_2km"])
print(f"\nDone.")

In [ ]:
# ============================================================
#  5. Ground Truth Validation (12 RL-Dispatch agg)
# ============================================================

disp = pd.read_csv(DISPATCH)
print(f"Dispatch ground truth: {len(disp):,} grids with real orders")

val = grid.drop(columns="geometry").merge(disp[["h3_id", "dispatch_orders"]], on="h3_id", how="inner")
val = val[val["dispatch_orders"] > 0].copy()
print(f"Matched grids with orders > 0: {len(val):,}")

check_cols = [
    ("takeout_demand_index", "Takeout Index (fused)"),
    ("real_order_count", "Real Order Count"),
    ("demand_pressure", "POI Demand (08)"),
    ("food_access_2km", "Food Access 2km"),
    ("pop_count", "Population"),
]

print("\n=== Correlation with dispatch_orders ===")
for col, label in check_cols:
    if col not in val.columns:
        continue
    r_p = val[col].corr(val["dispatch_orders"])
    r_s = val[col].corr(val["dispatch_orders"], method="spearman")
    print(f"  {label:30s}  Pearson r={r_p:.4f}  Spearman ρ={r_s:.4f}")

fig, axes = plt.subplots(1, len(check_cols), figsize=(5 * len(check_cols), 4))
for ax, (col, label) in zip(axes, check_cols):
    if col not in val.columns:
        continue
    ax.scatter(val[col], val["dispatch_orders"], alpha=0.3, s=8, c="#ff6b35")
    ax.set_xlabel(label, fontsize=9)
    ax.set_ylabel("Dispatch Orders")
    r = val[col].corr(val["dispatch_orders"])
    ax.set_title(f"r = {r:.3f}", fontsize=10)
    ax.grid(True, alpha=0.2)

plt.suptitle("Ground Truth: Fused Index vs Real Dispatch", fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(OUT / "validation_scatter.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {OUT / 'validation_scatter.png'}")

In [ ]:
# ============================================================
#  6. Save output
# ============================================================

out_cols = [
    "h3_id", "geometry",
    "real_order_count", "real_order_density",
    "pop_count", "xiaoqu_count", "food_count", "residential_count",
    "takeout_demand_index", "takeout_demand_norm",
    "food_access_1km", "food_access_2km", "food_access_3km",
    "food_access_norm",
]
out_cols = [c for c in out_cols if c in grid.columns]

out_gdf = grid[out_cols].copy()
out_path = OUT / "sz_takeout_demand_grid.gpkg"
out_gdf.to_file(out_path, driver="GPKG")

print(f"Written: {out_path}  ({len(out_gdf):,} rows)")
print(f"Columns: {[c for c in out_gdf.columns if c != 'geometry']}")
print(f"\n=== Summary ===")
print(f"Real orders: grids with >0 = {(grid['real_order_count']>0).sum():,}, "
      f"max={grid['real_order_count'].max():,.0f}")
print(f"Takeout index: mean={grid['takeout_demand_index'].mean():.4f}, "
      f"max={grid['takeout_demand_index'].max():.4f}")
print(f"Food access 2km: mean={grid['food_access_2km'].mean():.0f}, "
      f"max={grid['food_access_2km'].max():,}")